In [2]:
import os
from pathlib import Path
import pickle
import re
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import earthaccess

import numpy as np
import pandas as pd
import xarray as xr
import earthaccess
from tqdm.notebook import tqdm

In [3]:
PROJ_DIR = Path.cwd().parent.parent
DATA_DIR = PROJ_DIR / 'data'
lookup_fp = DATA_DIR / "lookup_station_times.pkl"
station_fp = DATA_DIR / 'station_coords.pkl'

In [4]:
# Load station -> [timestamps]
with open(lookup_fp, "rb") as f:
    lookup_station_times = pickle.load(f)

# Load station -> {lat, lon}
with open(station_fp, "rb") as f:
    station_coords = pickle.load(f)

# Quick diagnostics
stations_times = set(lookup_station_times.keys())
stations_coords = set(station_coords.keys())

missing_coords = sorted(stations_times - stations_coords)
missing_times  = sorted(stations_coords - stations_times)

print(f"Stations with times but no coords: {len(missing_coords)}")
if missing_coords[:10]:
    print("  examples:", missing_coords[:10])

print(f"Stations with coords but no times: {len(missing_times)}")
if missing_times[:10]:
    print("  examples:", missing_times[:10])

# Build the merged matchup driver table
rows = []
for station, times in lookup_station_times.items():
    if station not in station_coords:
        continue  # skip or handle separately

    lat = station_coords[station]["lat"]
    lon = station_coords[station]["lon"]

    for t in times:
        # Normalize to UTC pandas Timestamp
        t = pd.to_datetime(t, utc=True)
        rows.append({"station": station, "time": t, "lat": lat, "lon": lon})

stations_df = (
    pd.DataFrame(rows)
      .drop_duplicates(subset=["station", "time"])
      .sort_values(["station", "time"])
      .reset_index(drop=True)
)

stations_df.head(), stations_df.shape

Stations with times but no coords: 0
Stations with coords but no times: 0


(  station                      time      lat       lon
 0     BBB 2024-03-18 19:50:00+00:00  38.3126 -123.0825
 1     BBB 2024-05-16 18:30:00+00:00  38.3126 -123.0825
 2     BBB 2024-08-28 21:19:00+00:00  38.3126 -123.0825
 3     BBB 2024-09-30 16:11:00+00:00  38.3126 -123.0825
 4     BBB 2025-01-17 18:37:00+00:00  38.3126 -123.0825,
 (670, 4))

In [4]:
# 1. Login to Earthdata
auth = earthaccess.login()

In [6]:
def log_pace_granules_resumable_subset(
    df_subset,
    log_filename,
    time_window_hours=3,
    bbox_deg=1.0,
    flush_every_rows=1,
):
    """
    Resumable, append-only logger for a provided dataframe subset.
    Requirements:
      - df_subset has columns: lat, lon, time
      - df_subset has a stable integer column: row_id (0..n-1)
    Assumes earthaccess.login() already done by caller.
    """

    if "row_id" not in df_subset.columns:
        raise ValueError("df_subset must include a stable 'row_id' column (e.g., after reset_index).")

    done_rows = set()
    if os.path.exists(log_filename):
        prev = pd.read_csv(log_filename, usecols=["row_id", "status"])
        done_rows = set(prev.loc[prev["status"] == "ROW_DONE", "row_id"].unique())
        print(f"[resume] {os.path.basename(log_filename)}: {len(done_rows)} rows complete")

    write_header = not os.path.exists(log_filename)

    buffer = []
    rows_since_flush = 0

    for _, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc=os.path.basename(log_filename)):
        rid = int(row["row_id"])
        if rid in done_rows:
            continue

        lat0 = float(row["lat"])
        lon0 = float(row["lon"])
        t_insitu = pd.to_datetime(row["time"], utc=True)

        t0 = t_insitu - pd.Timedelta(hours=time_window_hours)
        t1 = t_insitu + pd.Timedelta(hours=time_window_hours)

        results = earthaccess.search_data(
            short_name="PACE_OCI_L2_AOP",
            temporal=(t0.to_pydatetime(), t1.to_pydatetime()),
            bounding_box=(lon0 - bbox_deg, lat0 - bbox_deg, lon0 + bbox_deg, lat0 + bbox_deg),
            cloud_hosted=True,
        )

        if len(results) == 0:
            buffer.append({
                "row_id": rid,
                "time": t_insitu,
                "lat": lat0,
                "lon": lon0,
                "granule_id": "NONE",
                "status": "No Granules Found",
            })
        else:
            for g in results:
                g_id = (
                    getattr(g, "producer_granule_id", None)
                    or getattr(g, "title", None)
                    or str(g)
                )

                dt = None
                remote = None
                status = None

                try:
                    remote = earthaccess.open([g], provider="OB_CLOUD")[0]
                    dt = xr.open_datatree(remote)

                    lat2d = dt["navigation_data"]["latitude"]
                    lon2d = dt["navigation_data"]["longitude"]

                    lat_min = float(lat2d.min().values)
                    lat_max = float(lat2d.max().values)
                    lon_min = float(lon2d.min().values)
                    lon_max = float(lon2d.max().values)

                    status = (
                        "Swath Covers Station"
                        if (lat_min <= lat0 <= lat_max and lon_min <= lon0 <= lon_max)
                        else "Swath Misses Station"
                    )

                except Exception as e:
                    status = f"Error: {type(e).__name__}: {e}"

                finally:
                    try:
                        if dt is not None:
                            dt.close()
                    except Exception:
                        pass
                    dt = None
                    remote = None

                buffer.append({
                    "row_id": rid,
                    "time": t_insitu,
                    "lat": lat0,
                    "lon": lon0,
                    "granule_id": g_id,
                    "status": status,
                })

        buffer.append({
            "row_id": rid,
            "time": t_insitu,
            "lat": lat0,
            "lon": lon0,
            "granule_id": "",
            "status": "ROW_DONE",
        })

        rows_since_flush += 1

        if rows_since_flush >= flush_every_rows:
            pd.DataFrame(buffer).to_csv(log_filename, mode="a", header=write_header, index=False)
            write_header = False
            buffer.clear()
            rows_since_flush = 0

    if buffer:
        pd.DataFrame(buffer).to_csv(log_filename, mode="a", header=write_header, index=False)

    return log_filename


In [6]:
stations_df.station.value_counts()

station
BML    104
CPP     98
SIO     93
SW      92
NP      74
MBB     73
MBF     73
TBB     18
T00     18
T16     17
BBB     10
Name: count, dtype: int64


---

SIO

In [7]:
station = 'SIO'
df_sta = stations_df[stations_df.station == station].copy()
df_sta = df_sta.reset_index(drop=True)
df_sta["row_id"] = df_sta.index

logfile = log_pace_granules_resumable_subset(
    df_sta,
    log_filename=f"{station}_pace_diagnostics.csv",
    time_window_hours=3,
    bbox_deg=1.0,
    flush_every_rows=1
)

[resume] SIO_pace_diagnostics.csv: 93 rows complete


SIO_pace_diagnostics.csv:   0%|          | 0/93 [00:00<?, ?it/s]

In [7]:
logfile = "SIO_pace_diagnostics.csv"
log = pd.read_csv(logfile)

done_rows = log.loc[log.status == "ROW_DONE", "row_id"]
last_done = int(done_rows.max()) if len(done_rows) else -1
next_row_id = last_done + 1

last_done, next_row_id


(49, 50)

In [8]:
station = "SIO"
df_sta = stations_df[stations_df.station == station].copy()
df_sta = df_sta.reset_index(drop=True)
df_sta["row_id"] = df_sta.index

df_sta.loc[df_sta.row_id == next_row_id, ["row_id", "time", "lat", "lon"]]


,row_id,time,lat,lon
50,50,2025-02-24 20:39:00+00:00,32.867,-117.257


In [11]:
# Pull the good granule names already coverd:
covers = log.loc[log.status == "Swath Covers Station", "granule_id"]
covers = covers[covers.notnull() & (covers != "NONE") & (covers != "")]
granules_useful = sorted(set(covers))

len(granules_useful), granules_useful[:10]


(55,
 ["Collection: {'ShortName': 'PACE_OCI_L2_AOP', 'Version': '3.1'}\nSpatial coverage: {'HorizontalSpatialDomain': {'Geometry': {'GPolygons': [{'Boundary': {'Points': [{'Latitude': 36.3242, 'Longitude': -107.57361}, {'Latitude': 30.86555, 'Longitude': -135.88306}, {'Latitude': 8.13026, 'Longitude': -128.43558}, {'Latitude': 13.13912, 'Longitude': -104.42388}, {'Latitude': 36.3242, 'Longitude': -107.57361}]}}]}}}\nTemporal coverage: {'RangeDateTime': {'EndingDateTime': '2024-04-15T20:45:17Z', 'BeginningDateTime': '2024-04-15T20:40:17Z'}}\nSize(MB): 158.6797866821289\nData: ['https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20240415T204017.L2.OC_AOP.V3_1.nc']",
  "Collection: {'ShortName': 'PACE_OCI_L2_AOP', 'Version': '3.1'}\nSpatial coverage: {'HorizontalSpatialDomain': {'Geometry': {'GPolygons': [{'Boundary': {'Points': [{'Latitude': 38.6699, 'Longitude': -109.1235}, {'Latitude': 33.13206, 'Longitude': -138.19722}, {'Latitude': 15.66324, 'Longitude': -131.

In [12]:
df_sta

,station,time,lat,lon,row_id
0,SIO,2024-03-11 19:15:00+00:00,32.867,-117.257,0
1,SIO,2024-03-18 19:00:00+00:00,32.867,-117.257,1
2,SIO,2024-03-25 19:34:00+00:00,32.867,-117.257,2
3,SIO,2024-04-01 19:27:00+00:00,32.867,-117.257,3
4,SIO,2024-04-08 19:53:00+00:00,32.867,-117.257,4
...,...,...,...,...,...
88,SIO,2025-11-17 20:01:00+00:00,32.867,-117.257,88
89,SIO,2025-12-01 20:49:00+00:00,32.867,-117.257,89
90,SIO,2025-12-08 20:17:00+00:00,32.867,-117.257,90
91,SIO,2025-12-15 20:06:00+00:00,32.867,-117.257,91


In [ ]:
logfile = log_pace_granules_resumable_subset(
    df_sta,
    log_filename="SIO_pace_diagnostics.csv",
    time_window_hours=3,
    bbox_deg=1.0,
    flush_every_rows=1
)

logfile


[resume] SIO_pace_diagnostics.csv: 50 rows complete


SIO_pace_diagnostics.csv:   0%|          | 0/93 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

...

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

In [8]:
# 3rd pass
station = "SIO"
df_sta = stations_df[stations_df.station == station].copy()
df_sta = df_sta.reset_index(drop=True)
df_sta["row_id"] = df_sta.index


logfile = f"{station}_pace_diagnostics.csv"
log = pd.read_csv(logfile)

done_rows = log.loc[log.status == "ROW_DONE", "row_id"]
last_done = int(done_rows.max()) if len(done_rows) else -1
next_row_id = last_done + 1

last_done, next_row_id

(83, 84)

In [9]:
df_sta.loc[df_sta.row_id == next_row_id, ["row_id", "time", "lat", "lon"]]

,row_id,time,lat,lon
84,84,2025-10-20 21:21:00+00:00,32.867,-117.257


In [10]:
# Pull the good granule names already coverd:
covers = log.loc[log.status == "Swath Covers Station", "granule_id"]
covers = covers[covers.notnull() & (covers != "NONE") & (covers != "")]
granules_useful = sorted(set(covers))

len(granules_useful), granules_useful[:10]

(100,
 ["Collection: {'ShortName': 'PACE_OCI_L2_AOP', 'Version': '3.1'}\nSpatial coverage: {'HorizontalSpatialDomain': {'Geometry': {'GPolygons': [{'Boundary': {'Points': [{'Latitude': 34.60976, 'Longitude': -106.07507}, {'Latitude': 29.17929, 'Longitude': -133.90425}, {'Latitude': 11.6349, 'Longitude': -128.06076}, {'Latitude': 16.75861, 'Longitude': -103.69022}, {'Latitude': 34.60976, 'Longitude': -106.07507}]}}]}}}\nTemporal coverage: {'RangeDateTime': {'BeginningDateTime': '2025-02-24T20:34:56Z', 'EndingDateTime': '2025-02-24T20:39:55Z'}}\nSize(MB): 445.04975414276123\nData: ['https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20250224T203456.L2.OC_AOP.V3_1.nc']",
  "Collection: {'ShortName': 'PACE_OCI_L2_AOP', 'Version': '3.1'}\nSpatial coverage: {'HorizontalSpatialDomain': {'Geometry': {'GPolygons': [{'Boundary': {'Points': [{'Latitude': 36.30452, 'Longitude': -100.69791}, {'Latitude': 30.84549, 'Longitude': -129.04016}, {'Latitude': 13.32932, 'Longitude':

In [11]:
logfile = log_pace_granules_resumable_subset(
    df_sta,
    log_filename=f"{station}_pace_diagnostics.csv",
    time_window_hours=3,
    bbox_deg=1.0,
    flush_every_rows=1
)

logfile

[resume] SIO_pace_diagnostics.csv: 84 rows complete


SIO_pace_diagnostics.csv:   0%|          | 0/93 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

...

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

'SIO_pace_diagnostics.csv'

In [12]:
log = pd.read_csv("SIO_pace_diagnostics.csv", usecols=["row_id","status"])
n_rows = log.loc[log.status == "ROW_DONE", "row_id"].nunique()
n_rows

93


---

BBB

In [13]:
station = 'BBB'
df_sta = stations_df[stations_df.station == station].copy()
df_sta = df_sta.reset_index(drop=True)
df_sta["row_id"] = df_sta.index

logfile = log_pace_granules_resumable_subset(
    df_sta,
    log_filename=f"{station}_pace_diagnostics.csv",
    time_window_hours=3,
    bbox_deg=1.0,
    flush_every_rows=1
)

BBB_pace_diagnostics.csv:   0%|          | 0/10 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

...

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

In [15]:
stations_df.station.value_counts()

station
BML    104
CPP     98
SIO     93
SW      92
NP      74
MBB     73
MBF     73
TBB     18
T00     18
T16     17
BBB     10
Name: count, dtype: int64


---

BML

In [ ]:
station = 'BML'
df_sta = stations_df[stations_df.station == station].copy()
df_sta = df_sta.reset_index(drop=True)
df_sta["row_id"] = df_sta.index

logfile = log_pace_granules_resumable_subset(
    df_sta,
    log_filename=f"{station}_pace_diagnostics.csv",
    time_window_hours=3,
    bbox_deg=1.0,
    flush_every_rows=1
)

BML_pace_diagnostics.csv:   0%|          | 0/104 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

...

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

In [7]:
station = 'BML'
log = pd.read_csv(f"{station}_pace_diagnostics.csv", usecols=["row_id","status"])
n_rows = log.loc[log.status == "ROW_DONE", "row_id"].nunique()
n_rows

86

In [8]:
df_sta = stations_df[stations_df.station == station].copy()
df_sta = df_sta.reset_index(drop=True)
df_sta["row_id"] = df_sta.index


logfile = f"{station}_pace_diagnostics.csv"
log = pd.read_csv(logfile)

done_rows = log.loc[log.status == "ROW_DONE", "row_id"]
last_done = int(done_rows.max()) if len(done_rows) else -1
next_row_id = last_done + 1

last_done, next_row_id

(85, 86)

In [9]:
logfile = log_pace_granules_resumable_subset(
    df_sta,
    log_filename=f"{station}_pace_diagnostics.csv",
    time_window_hours=3,
    bbox_deg=1.0,
    flush_every_rows=1
)

logfile

[resume] BML_pace_diagnostics.csv: 86 rows complete


BML_pace_diagnostics.csv:   0%|          | 0/104 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

...

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

'BML_pace_diagnostics.csv'


---

CPP


In [8]:
station = 'CPP'
df_sta = stations_df[stations_df.station == station].copy()
df_sta = df_sta.reset_index(drop=True)
df_sta["row_id"] = df_sta.index

logfile = log_pace_granules_resumable_subset(
    df_sta,
    log_filename=f"{station}_pace_diagnostics.csv",
    time_window_hours=3,
    bbox_deg=1.0,
    flush_every_rows=1
)

[resume] CPP_pace_diagnostics.csv: 98 rows complete


CPP_pace_diagnostics.csv:   0%|          | 0/98 [00:00<?, ?it/s]


---

SW

In [9]:
station = 'SW'
df_sta = stations_df[stations_df.station == station].copy()
df_sta = df_sta.reset_index(drop=True)
df_sta["row_id"] = df_sta.index

logfile = log_pace_granules_resumable_subset(
    df_sta,
    log_filename=f"{station}_pace_diagnostics.csv",
    time_window_hours=3,
    bbox_deg=1.0,
    flush_every_rows=1
)

SW_pace_diagnostics.csv:   0%|          | 0/92 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

In [10]:
station = 'NP'
df_sta = stations_df[stations_df.station == station].copy()
df_sta = df_sta.reset_index(drop=True)
df_sta["row_id"] = df_sta.index

logfile = log_pace_granules_resumable_subset(
    df_sta,
    log_filename=f"{station}_pace_diagnostics.csv",
    time_window_hours=3,
    bbox_deg=1.0,
    flush_every_rows=1
)

NP_pace_diagnostics.csv:   0%|          | 0/74 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

...

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

In [7]:
station = 'MBB'
df_sta = stations_df[stations_df.station == station].copy()
df_sta = df_sta.reset_index(drop=True)
df_sta["row_id"] = df_sta.index

logfile = log_pace_granules_resumable_subset(
    df_sta,
    log_filename=f"{station}_pace_diagnostics.csv",
    time_window_hours=3,
    bbox_deg=1.0,
    flush_every_rows=1
)

MBB_pace_diagnostics.csv:   0%|          | 0/73 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

...

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

In [11]:
station = 'MBF'
df_sta = stations_df[stations_df.station == station].copy()
df_sta = df_sta.reset_index(drop=True)
df_sta["row_id"] = df_sta.index

logfile = log_pace_granules_resumable_subset(
    df_sta,
    log_filename=f"{station}_pace_diagnostics.csv",
    time_window_hours=3,
    bbox_deg=1.0,
    flush_every_rows=1
)

MBF_pace_diagnostics.csv:   0%|          | 0/73 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

...

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

In [12]:
station = 'TBB'
df_sta = stations_df[stations_df.station == station].copy()
df_sta = df_sta.reset_index(drop=True)
df_sta["row_id"] = df_sta.index

logfile = log_pace_granules_resumable_subset(
    df_sta,
    log_filename=f"{station}_pace_diagnostics.csv",
    time_window_hours=3,
    bbox_deg=1.0,
    flush_every_rows=1
)

TBB_pace_diagnostics.csv:   0%|          | 0/18 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

...

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

In [13]:
station = 'T00'
df_sta = stations_df[stations_df.station == station].copy()
df_sta = df_sta.reset_index(drop=True)
df_sta["row_id"] = df_sta.index

logfile = log_pace_granules_resumable_subset(
    df_sta,
    log_filename=f"{station}_pace_diagnostics.csv",
    time_window_hours=3,
    bbox_deg=1.0,
    flush_every_rows=1
)

T00_pace_diagnostics.csv:   0%|          | 0/18 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

...

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

In [14]:
station = 'T16'
df_sta = stations_df[stations_df.station == station].copy()
df_sta = df_sta.reset_index(drop=True)
df_sta["row_id"] = df_sta.index

logfile = log_pace_granules_resumable_subset(
    df_sta,
    log_filename=f"{station}_pace_diagnostics.csv",
    time_window_hours=3,
    bbox_deg=1.0,
    flush_every_rows=1
)

T16_pace_diagnostics.csv:   0%|          | 0/17 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

...

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]